# Simple Linear Regression  

---

## Overview

In this notebook we train a **Simple Linear Regression** model from scratch using our custom
`SimpleLinearRegression` class backed by `GradientDescent`. We then evaluate it on the
**California Housing** dataset (1990 Census) by asking a single question:

> *Can **median income** alone predict the **median house value** of a census block?*

### What you will learn
- How gradient descent minimises Mean Squared Error (MSE) step-by-step
- How to interpret the learned weight $w$ and bias $b$ in a real-world context
- How to assess fit quality with $R^2$ and a Predicted vs Actual plot

### Dataset
| Column | Description |
|---|---|
| `longitude` / `latitude` | Block group centroid |
| `housing_median_age` | Median age of houses in the block |
| `total_rooms` | Total rooms across all households |
| `total_bedrooms` | Total bedrooms across all households |
| `population` | Block group population |
| `households` | Number of households |
| `median_income` | Median income (in tens of thousands of USD) |
| **`median_house_value`** | **Target — median house value (USD)** |
| `ocean_proximity` | Categorical proximity to ocean |

*Source: [Kaggle — California Housing Data (1990)](https://www.kaggle.com/datasets/harrywang/housing)*

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, r'/Jana CMOR/2026_Data_Science_and_Machine_Learning/src/rice_ml/supervised_learning')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Our custom implementations
from rice_ml.supervised_learning.gradient_descent import GradientDescent
from rice_ml.supervised_learning.linear_regression import SimpleLinearRegression

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.35,
})
print('All imports OK.')

## 2. Load & Inspect the Data

Download `housing.csv` from [Kaggle](https://www.kaggle.com/datasets/harrywang/housing) and
place it in the same directory as this notebook.

In [ ]:
df = pd.read_csv('housing.csv')
print(f'Shape: {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)')
df.head()

In [ ]:
# Quick summary statistics
df[['median_income', 'median_house_value']].describe().round(2)

In [ ]:
# Missing value audit
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

## 3. The Regression Neuron

A **simple linear regression** can be thought of as a single neuron with no activation function.
It maps one input feature $x$ to a continuous output $\hat{y}$ via:

$$\hat{y} = w \cdot x + b$$

where $w$ is the **weight** (slope) and $b$ is the **bias** (intercept).

### Loss — Mean Squared Error

We measure how wrong the model is using **MSE**:

$$\mathcal{L}(w, b) = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2$$

### Gradient Descent Updates

We iteratively nudge $w$ and $b$ in the direction that reduces $\mathcal{L}$:

$$\frac{\partial \mathcal{L}}{\partial w} = -\frac{2}{n} \sum_{i=1}^{n} x_i \left(y_i - \hat{y}_i\right)$$

$$\frac{\partial \mathcal{L}}{\partial b} = -\frac{2}{n} \sum_{i=1}^{n} \left(y_i - \hat{y}_i\right)$$

$$w \leftarrow w - \alpha \frac{\partial \mathcal{L}}{\partial w}, \qquad b \leftarrow b - \alpha \frac{\partial \mathcal{L}}{\partial b}$$

where $\alpha$ is the **learning rate** — how large each step is.

## 4. Preprocessing

We select **`median_income`** as our single feature $x$ and **`median_house_value`** as the
target $y$, then apply a simple train / test split and **standardise** $x$ so that gradient
descent converges smoothly regardless of the learning rate chosen.

In [ ]:
# Drop rows with any NaN (affects total_bedrooms; our two columns are complete)
df_clean = df[['median_income', 'median_house_value']].dropna().reset_index(drop=True)
print(f'Clean rows: {len(df_clean):,}')

X_raw = df_clean['median_income'].values       # shape (n,)
y_raw = df_clean['median_house_value'].values   # shape (n,)

In [ ]:
# ── Train / Test split (80 / 20, no shuffle for reproducibility) ──
split = int(0.8 * len(X_raw))
X_train_raw, X_test_raw = X_raw[:split], X_raw[split:]
y_train,      y_test      = y_raw[:split], y_raw[split:]

# ── Standardise X: z = (x - μ) / σ  (fit on train only) ──
mu    = X_train_raw.mean()
sigma = X_train_raw.std()

X_train = (X_train_raw - mu) / sigma
X_test  = (X_test_raw  - mu) / sigma

print(f'Train size : {len(X_train):,}')
print(f'Test  size : {len(X_test):,}')
print(f'Income μ={mu:.4f}  σ={sigma:.4f}')

## 5. Exploratory Data Analysis

Before training, it's always worth visualising the relationship between feature and target.
A positive linear trend here would confirm that `median_income` is a reasonable predictor.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(X_train_raw, y_train / 1e5,
           alpha=0.25, s=8, color='steelblue', label='Train')
ax.scatter(X_test_raw,  y_test  / 1e5,
           alpha=0.25, s=8, color='tomato',    label='Test')

ax.set_xlabel('Median Income (tens of thousands USD)')
ax.set_ylabel('Median House Value (×$100k)')
ax.set_title('California Housing — Income vs House Value')
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## 6. Training

We instantiate `SimpleLinearRegression`, which internally builds a `GradientDescent`
optimiser and injects the MSE cost and gradient functions.

**Hyperparameters to try:**
| Parameter | Meaning | Default used |
|---|---|---|
| `alpha` | Learning rate $\alpha$ | `0.01` |
| `max_iter` | Max gradient descent steps | `3000` |
| `tol` | Early-stop when cost change $< $ tol | `1e-6` |
| `random_state` | Seed for weight init | `42` |

In [ ]:
model = SimpleLinearRegression(
    alpha=0.01,
    max_iter=3000,
    tol=1e-6,
    random_state=42,
)

model.fit(X_train, y_train)
print(model)
print(f'\nTraining stopped at epoch {len(model.cost_history_):,}')

In [ ]:
# Interpret the learned parameters in original (un-standardised) units
# Since X_std = (X_raw - mu) / sigma, the original-scale weight is:
w_orig = model.coef_ / sigma
b_orig = model.intercept_ - model.coef_ * mu / sigma

print('Learned model (standardised feature space):')
print(f'  ŷ = {model.coef_:.2f} · x_std + {model.intercept_:.2f}')
print()
print('Equivalent model in original feature space:')
print(f'  ŷ ≈ ${w_orig:,.0f} × median_income + ${b_orig:,.0f}')
print()
print(f'Interpretation: each additional $10k of median income is associated')
print(f'with ~${w_orig:,.0f} higher median house value.')

## 7. Training Cost Curve

The cost curve shows how MSE decreases each epoch. A smooth, monotonically
decreasing curve confirms that the learning rate $\alpha$ is well-chosen.
A curve that oscillates or explodes upward signals $\alpha$ is too large.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Full curve
axes[0].plot(model.cost_history_, color='steelblue', linewidth=1.5)
axes[0].set_title('MSE — All Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')

# Zoom: first 200 epochs to see the steep initial drop
axes[1].plot(model.cost_history_[:200], color='tomato', linewidth=1.5)
axes[1].set_title('MSE — First 200 Epochs (zoom)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')

plt.suptitle('Gradient Descent Cost Curve', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Evaluation

We evaluate on the **held-out test set** using:

| Metric | Formula | Interpretation |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(y - \hat{y})^2$ | Average squared error (in $\text{USD}^2$) |
| **RMSE** | $\sqrt{\text{MSE}}$ | Average error in USD — same units as target |
| **$R^2$** | $1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}$ | Fraction of variance explained (1 = perfect) |

In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test  = model.predict(X_test)

def rmse(y_true, y_hat):
    return float(np.sqrt(np.mean((y_true - y_hat) ** 2)))

metrics = {
    'Split':    ['Train', 'Test'],
    'MSE':      [np.mean((y_train - y_pred_train)**2),
                 np.mean((y_test  - y_pred_test )**2)],
    'RMSE':     [rmse(y_train, y_pred_train),
                 rmse(y_test,  y_pred_test)],
    'R²':       [model.score(X_train, y_train),
                 model.score(X_test,  y_test)],
}

pd.DataFrame(metrics).set_index('Split').round(4)

## 9. Predicted vs Actual Plot

The **Predicted vs Actual** plot is the primary diagnostic for a regression model.

- Points on the **diagonal dashed line** ($\hat{y} = y$) represent perfect predictions.
- Points **above** the line: model *underestimates* the true value.
- Points **below** the line: model *overestimates* the true value.
- A systematic fan or curve reveals heteroscedasticity or non-linearity respectively.

We also show a **Residual Distribution** to check whether errors are approximately
Gaussian and centred on zero — a key assumption of OLS.

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# ── Left: Predicted vs Actual ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])

ax1.scatter(
    y_test / 1e5, y_pred_test / 1e5,
    alpha=0.3, s=10, color='steelblue', label='Test samples'
)

# Perfect-prediction diagonal
lim_lo = min(y_test.min(), y_pred_test.min()) / 1e5
lim_hi = max(y_test.max(), y_pred_test.max()) / 1e5
ax1.plot([lim_lo, lim_hi], [lim_lo, lim_hi],
         color='tomato', linewidth=1.8, linestyle='--', label='Perfect fit')

ax1.set_xlabel('Actual Median House Value (×$100k)', fontsize=11)
ax1.set_ylabel('Predicted Median House Value (×$100k)', fontsize=11)
ax1.set_title('Predicted vs Actual\n(Test Set)', fontsize=12, fontweight='bold')
ax1.legend(markerscale=3, fontsize=9)

r2_text = f'$R^2 = {model.score(X_test, y_test):.3f}$'
ax1.text(0.05, 0.92, r2_text, transform=ax1.transAxes, fontsize=10,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.6))

# ── Right: Residual Distribution ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])

residuals = y_test - y_pred_test
ax2.hist(residuals / 1e5, bins=60, color='steelblue', edgecolor='white',
         linewidth=0.4, alpha=0.85)
ax2.axvline(0, color='tomato', linewidth=1.8, linestyle='--', label='Zero error')
ax2.axvline(residuals.mean() / 1e5, color='gold', linewidth=1.8,
            linestyle='-', label=f'Mean error = ${residuals.mean():,.0f}')

ax2.set_xlabel('Residual (×$100k)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('Residual Distribution\n(Test Set)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

plt.suptitle(
    'Simple Linear Regression — Median Income → Median House Value',
    fontsize=13, fontweight='bold', y=1.02
)
plt.savefig('predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to predicted_vs_actual.png')

## 10. Regression Line Over Raw Data

Plotting the fitted line on top of the raw scatter gives an intuitive view of
what the model has learned: a straight line that minimises the sum of squared
vertical distances to every point.

In [ ]:
# Build a smooth regression line in original-scale income space
x_line_raw = np.linspace(X_train_raw.min(), X_train_raw.max(), 300)
x_line_std = (x_line_raw - mu) / sigma
y_line     = model.predict(x_line_std)

fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(X_train_raw, y_train / 1e5,
           alpha=0.2, s=8, color='steelblue', label='Train data')
ax.scatter(X_test_raw, y_test / 1e5,
           alpha=0.2, s=8, color='lightcoral', label='Test data')
ax.plot(x_line_raw, y_line / 1e5,
        color='tomato', linewidth=2.5,
        label=f'Fit: $\\hat{{y}} \\approx$ ${w_orig:,.0f} × income + ${b_orig:,.0f}')

ax.set_xlabel('Median Income (tens of thousands USD)', fontsize=11)
ax.set_ylabel('Median House Value (×$100k)', fontsize=11)
ax.set_title('Fitted Regression Line — California Housing', fontsize=12, fontweight='bold')
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

## 11. Discussion & Limitations

### What the model learned
The gradient descent optimiser successfully minimised MSE and learned a positive
linear relationship: higher median income → higher median house value. The slope
is interpretable — each unit increase in `median_income` (roughly $10k)
corresponds to an estimated increase of ~$40–50k in house value.

### Why $R^2$ is moderate (~0.45–0.50)
Income alone explains roughly half the variance in house prices. The remainder
is driven by factors our single-feature model ignores:

- **Location** (`latitude`, `longitude`) — proximity to the coast or a city centre
- **Housing stock** (`housing_median_age`, `total_rooms`) — older neighbourhoods
  may be undervalued
- **Density** (`population`, `households`) — crowded blocks often have lower values
- **Ocean proximity** — the categorical `ocean_proximity` feature is highly
  predictive

### The price cap at $500,000
A horizontal band of points at $500k is visible in the scatter plots. The original
dataset **censored** house values above this threshold by clamping them to $500k.
This creates a systematic underestimate for our model on high-income blocks.

### Next steps
| Improvement | Expected gain |
|---|---|
| Add more numeric features → **Multiple Linear Regression** | $R^2$ up to ~0.65 |
| One-hot encode `ocean_proximity` | Further improvement |
| Feature engineering (e.g. `rooms_per_household`) | Modest gain |
| Non-linear model (e.g. polynomial, random forest) | Significant gain |